# Lab 11: CNN Model for Time-Series Forecasting

**Student Name:** Samiullah Khan  
**Registration Number:** 22JZELE0492  
**Course:** Machine Learning Lab  
**Supervisor:** Engr. Irshad Ullah  
**University:** UET Peshawar - Nowshera Campus  

## Lab Overview
This notebook develops a one-dimensional CNN model for time-series forecasting. It defines a `Conv1D` architecture, sets up checkpoints and training callbacks, loads preprocessed time-series data, trains the model, and evaluates forecasting performance.

## Learning Objectives
- Import TensorFlow/Keras, Scikit-learn metrics, and custom time-series utilities.
- Define a CNN architecture using `Conv1D` layers for sequential data.
- Configure callbacks, optimizer, checkpoints, and training paths.
- Load train, validation, and test datasets for forecasting.
- Evaluate CNN forecasting performance using standard regression metrics.

## Section 1: Working Directory and Library Imports
This section sets the Lab 10/11 path and imports all libraries required for CNN-based time-series forecasting.


In [1]:
# Set the working directory for Lab 10/11 files.
import os
os.chdir(r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11')

In [2]:
# Import metrics, time-series helpers, callbacks, TensorFlow/Keras layers, and utilities.
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [3]:
# Define training state and input dimensions for the time-series windows.
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [4]:
# Build a Conv1D CNN model for time-series forecasting.
def CNN():
    input_data = Input(shape=(time_steps, num_features))
    x1 = Conv1D(16, 2, activation="relu")(input_data)
    x2 = Conv1D(16, 2, activation="relu")(x1)
    flatten = Flatten()(x2)
    output_data = Dense(1)(flatten)
    model = Model(input_data, output_data)
    return model

In [5]:
model1 = CNN()
model1.summary()

Model: "functional"

â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”³â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”³â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”“
â”ƒ Layer (type)                    â”ƒ Output Shape           â”ƒ       Param # â”ƒ
â”¡â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â•‡â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â•‡â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”©
â”‚ input_layer (InputLayer)        â”‚ (None, 24, 21)         â”‚             0 â”‚
â”œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¼â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¼â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¤
â”‚ conv1d (Conv1D)                 â”‚ (None, 23, 16)         â”‚           688 â”‚
â”œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¼â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¼â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¤
â”‚ conv1d_1 (Conv1D)               â”‚ (None, 22, 16)         â”‚           528 â”‚
â”œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¼â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¼â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¤
â”‚ flatten (Flatten)               â”‚ (None, 352)            â”‚             0 â”‚
â”œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¼â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¼â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¤
â”‚ dense (Dense)                   â”‚ (None, 1)              â”‚           353 â”‚
â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”´â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”´â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”˜

 Total params: 1,569 (6.13 KB)

 Trainable params: 1,569 (6.13 KB)

 Non-trainable params: 0 (0.00 B)

## Section 2: Model Parameters and CNN Architecture
The following cells define input shape parameters and build the Conv1D-based CNN model for forecasting.


In [6]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [7]:
# Define checkpoint, output, figure, and training-history paths.
checkpoints = r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [8]:
# Save only the best model checkpoint based on validation loss.
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# Group callbacks for model training.
callbacks = [EpochCheckpoint1,TrainingMonitor1]

In [9]:
# Compile a new CNN model or load an existing checkpoint for continued training.
if model is None:
    print("[INFO] compiling model...")
    model =CNN()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # Update the learning rate before resuming training.
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


In [10]:
# Load train, validation, test, and scaler files.
import os
path_dataset =r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\Sami\anaconda3\envs\machinelearning\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

## Section 3: Training Configuration and Callback Setup
This section prepares checkpoint saving, monitoring, optimizer configuration, and model compilation for training.


In [11]:
# Confirm the forecasting window size and feature count.
time_steps=24
num_features=21

In [12]:
# Convert raw arrays into univariate multi-step forecasting windows.
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.008221149444580078 sec


In [13]:
# Train the CNN model on the prepared forecasting windows.
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,verbose = verbose)

Epoch 1/10
20/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 6ms/step - loss: 0.2924 - mae: 0.2924 - mape: 179.7408
Epoch 1: val_loss improved from None to 0.07931, saving model to C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E1-cp-0001-loss0.08.h5


27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 3s 38ms/step - loss: 0.1887 - mae: 0.1887 - mape: 101.3916 - val_loss: 0.0793 - val_mae: 0.0793 - val_mape: 25.3770
Epoch 2/10
16/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 7ms/step - loss: 0.0784 - mae: 0.0784 - mape: 33.2964 
Epoch 2: val_loss improved from 0.07931 to 0.06516, saving model to C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E1-cp-0002-loss0.07.h5


27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 1s 25ms/step - loss: 0.0774 - mae: 0.0774 - mape: 38.4661 - val_loss: 0.0652 - val_mae: 0.0652 - val_mape: 22.0145
Epoch 3/10
20/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 6ms/step - loss: 0.0671 - mae: 0.0671 - mape: 39.8672
Epoch 3: val_loss improved from 0.06516 to 0.06168, saving model to C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E1-cp-0003-loss0.06.h5


27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 1s 23ms/step - loss: 0.0659 - mae: 0.0659 - mape: 37.4440 - val_loss: 0.0617 - val_mae: 0.0617 - val_mape: 20.7757
Epoch 4/10
20/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 6ms/step - loss: 0.0579 - mae: 0.0579 - mape: 29.1834
Epoch 4: val_loss improved from 0.06168 to 0.05319, saving model to C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E1-cp-0004-loss0.05.h5


27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 1s 20ms/step - loss: 0.0569 - mae: 0.0569 - mape: 27.1861 - val_loss: 0.0532 - val_mae: 0.0532 - val_mape: 18.6455
Epoch 5/10
27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 6ms/step - loss: 0.0494 - mae: 0.0494 - mape: 20.4357
Epoch 5: val_loss did not improve from 0.05319
27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 1s 19ms/step - loss: 0.0507 - mae: 0.0507 - mape: 22.0388 - val_loss: 0.0707 - val_mae: 0.0707 - val_mape: 24.0913
Epoch 6/10
19/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 6ms/step - loss: 0.0477 - mae: 0.0477 - mape: 18.5259
Epoch 6: val_loss improved from 0.05319 to 0.04988, saving model to C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E1-cp-0006-loss0.05.h5


27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 1s 32ms/step - loss: 0.0445 - mae: 0.0445 - mape: 18.6698 - val_loss: 0.0499 - val_mae: 0.0499 - val_mape: 16.9805
Epoch 7/10
22/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 8ms/step - loss: 0.0413 - mae: 0.0413 - mape: 15.8919
Epoch 7: val_loss did not improve from 0.04988
27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 1s 38ms/step - loss: 0.0430 - mae: 0.0430 - mape: 18.1860 - val_loss: 0.0622 - val_mae: 0.0622 - val_mape: 22.0590
Epoch 8/10
23/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 11ms/step - loss: 0.0423 - mae: 0.0423 - mape: 16.5030
Epoch 8: val_loss did not improve from 0.04988
27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 2s 51ms/step - loss: 0.0404 - mae: 0.0404 - mape: 17.5447 - val_loss: 0.0532 - val_mae: 0.0532 - val_mape: 18.3466
Epoch 9/10
18/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 6ms/step

In [14]:
# Load the trained CNN model and evaluate forecasting performance on the test set.
model = load_model(r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E1-cp-0006-loss0.05.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squared Error (RMSE)
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 185ms/step
Mean Absolute Error (MAE): 8933.89
Median Absolute Error (MedAE): 8758.26
Mean Squared Error (MSE): 80371910.61
Root Mean Squared Error (RMSE): 8965.04
Mean Absolute Percentage Error (MAPE): 56.99 %
Median Absolute Percentage Error (MDAPE): 56.59 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


In [15]:
# Configure fine-tuning checkpoint path and starting model.
checkpoints = r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E1-cp-0006-loss0.05.h5'
model=r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E1-cp-0006-loss0.05.h5'
start_epoch= 58

## Section 4: Dataset Loading, Prediction, and Evaluation
The remaining cells load the data, prepare it for the CNN model, generate predictions, and calculate forecasting metrics.


In [16]:
# Set up fine-tuning callbacks and load the previous CNN checkpoint.
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# Group callbacks for continued training.
callbacks = [EpochCheckpoint1,TrainingMonitor1]
model_path = r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E1-cp-0006-loss0.05.h5'
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
else:
    print("[INFO] loading {}...".format(model_path))
    model = load_model(model_path, compile=False)

    opt = Adam(learning_rate=1e-4)
    model.compile(
        loss="mae",
        optimizer=opt,
        metrics=["mae", "mape"]
    )

    # Display the learning rate used for fine tuning.
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))

[INFO] loading C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E1-cp-0006-loss0.05.h5...
[INFO] old learning rate: 9.999999747378752e-05
[INFO] new learning rate: 9.999999747378752e-05


In [17]:
# Continue training the loaded CNN model.
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
21/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 5ms/step - loss: 0.0399 - mae: 0.0399 - mape: 17.8094
Epoch 1: val_loss improved from None to 0.06041, saving model to C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E1-cp-0006-loss0.05.h5


27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 3s 37ms/step - loss: 0.0402 - mae: 0.0402 - mape: 17.1266 - val_loss: 0.0604 - val_mae: 0.0604 - val_mape: 20.9160
Epoch 2/10
27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 6ms/step - loss: 0.0386 - mae: 0.0386 - mape: 15.5628
Epoch 2: val_loss improved from 0.06041 to 0.05975, saving model to C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E1-cp-0006-loss0.05.h5


27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 1s 30ms/step - loss: 0.0395 - mae: 0.0395 - mape: 16.3133 - val_loss: 0.0597 - val_mae: 0.0597 - val_mape: 20.8247
Epoch 3/10
27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 6ms/step - loss: 0.0397 - mae: 0.0397 - mape: 17.0770 
Epoch 3: val_loss did not improve from 0.05975
27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 1s 22ms/step - loss: 0.0389 - mae: 0.0389 - mape: 15.9712 - val_loss: 0.0608 - val_mae: 0.0608 - val_mape: 21.2534
Epoch 4/10
20/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 6ms/step - loss: 0.0374 - mae: 0.0374 - mape: 14.8485 
Epoch 4: val_loss did not improve from 0.05975
27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 1s 23ms/step - loss: 0.0384 - mae: 0.0384 - mape: 15.8734 - val_loss: 0.0708 - val_mae: 0.0708 - val_mape: 24.6618
Epoch 5/10
27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 6ms/ste

27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 1s 24ms/step - loss: 0.0375 - mae: 0.0375 - mape: 16.3068 - val_loss: 0.0597 - val_mae: 0.0597 - val_mape: 20.7607
Epoch 9/10
25/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 6ms/step - loss: 0.0367 - mae: 0.0367 - mape: 17.3047
Epoch 9: val_loss did not improve from 0.05968
27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 1s 29ms/step - loss: 0.0368 - mae: 0.0368 - mape: 15.9654 - val_loss: 0.0619 - val_mae: 0.0619 - val_mape: 21.4110
Epoch 10/10
22/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 9ms/step - loss: 0.0355 - mae: 0.0355 - mape: 16.2256
Epoch 10: val_loss did not improve from 0.05968
27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 1s 29ms/step - loss: 0.0364 - mae: 0.0364 - mape: 15.6997 - val_loss: 0.0634 - val_mae: 0.0634 - val_mape: 22.1477


In [18]:
# Load the saved CNN model again and evaluate forecasting performance.
model = load_model(r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E1-cp-0006-loss0.05.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squared Error (RMSE)
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 179ms/step
Mean Absolute Error (MAE): 9498.7
Median Absolute Error (MedAE): 9342.16
Mean Squared Error (MSE): 90700801.12
Root Mean Squared Error (RMSE): 9523.7
Mean Absolute Percentage Error (MAPE): 60.61 %
Median Absolute Percentage Error (MDAPE): 60.37 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


## Final Conclusion
In this lab, a CNN model was applied to time-series forecasting. The notebook shows how `Conv1D` layers can extract local temporal patterns and how the trained model is evaluated using regression metrics.

## Submitted By
**Student Name:** Samiullah Khan  
**Registration Number:** 22JZELE0492  
**Course:** Machine Learning Lab  
**Supervisor:** Engr. Irshad Ullah  
**University:** UET Peshawar - Nowshera Campus